# 06. Phân tích hồi quy (D2 - D5)

Notebook này kiểm tra mối liên hệ giữa kích thước đoạn code, ngôn ngữ, công cụ AI và vai trò file với thời gian đến lần sửa đầu tiên, tỷ lệ sửa trong 30 ngày và khả năng phát sinh sửa lỗi (`bug_fix`). Tất cả mô hình đều dùng sai số chuẩn theo cụm `repo`.

Import thư viện, tải dữ liệu và tạo các biến logarit cần thiết (`log_code_lines`, `log_days`).

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from patsy import dmatrices

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

DATA_PATH = Path("../data/processed/commits_features.csv")
REPORT_DIR = Path("../reports")
FIG_DIR = REPORT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH, low_memory=False).replace([np.inf, -np.inf], np.nan)
df["days_to_modification"] = pd.to_numeric(df["days_to_modification"], errors="coerce")
df["log_code_lines"] = np.log1p(df["code_lines"])
df["log_days"] = np.log1p(df["days_to_modification"])
df["has_function_def"] = df["has_function_def"].astype(int)
df["has_class_def"] = df["has_class_def"].astype(int)

def cluster_kw(d):
    return {"cov_type": "cluster", "cov_kwds": {"groups": pd.factorize(d["repo"])[0]}}

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / name, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()

print(f"Loaded {len(df):,} rows x {df.shape[1]} columns")

Loaded 21,287 rows x 33 columns


## D2. Hồi quy tuyến tính bội trên `log1p(days_to_modification)`

Sử dụng biến logarit cho cả `days_to_modification` và `code_lines` để xử lý phân phối lệch phải, bổ sung thêm `file_role` và các cờ cấu trúc code.

In [2]:
d2_cols = ["log_days", "log_code_lines", "language", "ai_tool", "file_role", "has_function_def", "has_class_def", "repo"]
d2 = df[d2_cols].dropna().copy()

d2_formula = "log_days ~ log_code_lines + C(language) + C(ai_tool) + C(file_role) + has_function_def + has_class_def"
d2_model = smf.ols(d2_formula, data=d2).fit(**cluster_kw(d2))
print(d2_model.summary())

d2_result = pd.DataFrame({
    "term": d2_model.params.index,
    "coef": d2_model.params.values,
    "p_value": d2_model.pvalues.values,
    "ci_lower": d2_model.conf_int()[0].values,
    "ci_upper": d2_model.conf_int()[1].values,
})
d2_result.to_csv(REPORT_DIR / "regression_d2_multiple.csv", index=False)

plt.figure(figsize=(9, max(5, len(d2_result) * 0.28)))
coef = d2_result[d2_result["term"] != "Intercept"].sort_values("coef")
plt.errorbar(coef["coef"], range(len(coef)), xerr=[coef["coef"] - coef["ci_lower"], coef["ci_upper"] - coef["coef"]], fmt="o")
plt.yticks(range(len(coef)), coef["term"])
plt.axvline(0, color="black", lw=1)
plt.title("D2: He so hoi quy tren log1p(days_to_modification)")
plt.xlabel("He so (95% CI)")
savefig("D2_coefficient_plot.png")

                            OLS Regression Results                            
Dep. Variable:               log_days   R-squared:                       0.072
Model:                            OLS   Adj. R-squared:                  0.070
Method:                 Least Squares   F-statistic:                     9.736
Date:                Tue, 17 Nov 2026   Prob (F-statistic):           1.13e-21
Time:                        00:45:40   Log-Likelihood:                -10908.
No. Observations:                5714   AIC:                         2.185e+04
Df Residuals:                    5699   BIC:                         2.195e+04
Df Model:                          14                                         
Covariance Type:              cluster                                         
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Inte

## D3. Hồi quy Logistic dự đoán `change_30d`

Dùng biến mục tiêu `change_30d` (đã loại bỏ sai lệch do thời gian theo dõi không đều) và mã hóa `year` thành biến phân loại (`C(year)`) để tránh sai số số học ở hệ số chặn.

In [3]:
d3_cols = ["change_30d", "log_code_lines", "language", "ai_tool", "file_role", "ai_verb", "year", "repo"]
d3 = df[d3_cols].dropna().copy()
d3["change_30d"] = d3["change_30d"].astype(int)
d3["year"] = d3["year"].astype(int)

d3_formula = "change_30d ~ log_code_lines + C(language) + C(ai_tool) + C(file_role) + C(ai_verb) + C(year)"
d3_model = smf.glm(d3_formula, data=d3, family=sm.families.Binomial()).fit(**cluster_kw(d3))
print(d3_model.summary())

d3_result = pd.DataFrame({
    "term": d3_model.params.index,
    "log_odds_coef": d3_model.params.values,
    "odds_ratio": np.exp(d3_model.params.values),
    "p_value": d3_model.pvalues.values,
    "ci_lower_or": np.exp(d3_model.conf_int()[0].values),
    "ci_upper_or": np.exp(d3_model.conf_int()[1].values),
})
d3_result.to_csv(REPORT_DIR / "regression_d3_logistic.csv", index=False)

plt.figure(figsize=(9, max(5, len(d3_result) * 0.28)))
coef3 = d3_result[d3_result["term"] != "Intercept"].sort_values("odds_ratio")
plt.errorbar(coef3["odds_ratio"], range(len(coef3)), xerr=[coef3["odds_ratio"] - coef3["ci_lower_or"], coef3["ci_upper_or"] - coef3["odds_ratio"]], fmt="o")
plt.yticks(range(len(coef3)), coef3["term"])
plt.axvline(1, color="black", lw=1)
plt.xscale("log")
plt.title("D3: Odds ratio du doan change_30d")
plt.xlabel("Odds ratio (95% CI)")
savefig("D3_logistic_coefficients.png")

                 Generalized Linear Model Regression Results                  
Dep. Variable:             change_30d   No. Observations:                19554
Model:                            GLM   Df Residuals:                    19536
Model Family:                Binomial   Df Model:                           17
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -10331.
Date:                Tue, 17 Nov 2026   Deviance:                       20663.
Time:                        00:45:46   Pearson chi2:                 1.95e+04
No. Iterations:                     4   Pseudo R-squ. (CS):            0.02156
Covariance Type:              cluster                                         
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Inte

## D3b. Hồi quy Logistic dự đoán khả năng sửa lỗi (`is_bug_fix`)

Trên nhóm các khối code có ghi nhận thay đổi, kiểm tra yếu tố nào làm tăng khả năng lần sửa đầu tiên là sửa lỗi (`bug_fix`).

In [4]:
changed = df[df["has_recorded_change"] == True].copy()
if len(set(changed["change_intent"].unique()) - {"other_change", "no_recorded_change"}) == 0:
    print("Bo qua D3b vi chua co cot change_commit_message tu notebook 01.")
else:
    d3b = changed[["change_intent", "log_code_lines", "language", "ai_tool", "file_role", "has_function_def", "repo"]].dropna().copy()
    d3b["is_bug_fix"] = (d3b["change_intent"] == "bug_fix").astype(int)
    d3b_formula = "is_bug_fix ~ log_code_lines + C(language) + C(ai_tool) + C(file_role) + has_function_def"
    d3b_model = smf.glm(d3b_formula, data=d3b, family=sm.families.Binomial()).fit(**cluster_kw(d3b))
    print(d3b_model.summary())
    d3b_result = pd.DataFrame({
        "term": d3b_model.params.index,
        "log_odds_coef": d3b_model.params.values,
        "odds_ratio": np.exp(d3b_model.params.values),
        "p_value": d3b_model.pvalues.values,
    })
    d3b_result.to_csv(REPORT_DIR / "regression_d3b_bug_fix.csv", index=False)

                 Generalized Linear Model Regression Results                  
Dep. Variable:             is_bug_fix   No. Observations:                 7445
Model:                            GLM   Df Residuals:                     7431
Model Family:                Binomial   Df Model:                           13
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -3972.8
Date:                Tue, 17 Nov 2026   Deviance:                       7945.7
Time:                        00:45:49   Pearson chi2:                 7.46e+03
No. Iterations:                     5   Pseudo R-squ. (CS):            0.06775
Covariance Type:              cluster                                         
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Inte

## D4. Tóm tắt mô hình và kiểm tra đa cộng tuyến (VIF)

Lưu bảng đánh giá `R-squared`, `AIC` và tính chỉ số VIF cho các biến đầu vào của mô hình D2.

In [5]:
model_summary = pd.DataFrame([
    {"model": "D2 OLS log_days", "n": int(d2_model.nobs), "r_squared": d2_model.rsquared, "adj_r_squared": d2_model.rsquared_adj, "aic": d2_model.aic},
    {"model": "D3 GLM change_30d", "n": int(d3_model.nobs), "r_squared": np.nan, "adj_r_squared": np.nan, "aic": d3_model.aic},
])
display(model_summary)
model_summary.to_csv(REPORT_DIR / "regression_model_summary.csv", index=False)

_, X = dmatrices(d2_formula, data=d2, return_type="dataframe")
vif = pd.DataFrame({"term": X.columns, "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]})
display(vif)
vif.to_csv(REPORT_DIR / "regression_d2_vif.csv", index=False)

,model,n,r_squared,adj_r_squared,aic
0,D2 OLS log_days,5714,0.072091,0.069812,21846.592534
1,D3 GLM change_30d,19554,NaN,NaN,20698.718180


,term,VIF
0,Intercept,1.000000
1,C(language)[T.python],1.400990
2,C(ai_tool)[T.claude],1.545050
3,C(ai_tool)[T.gemini_bard],1.124668
4,C(ai_tool)[T.github_copilot],1.256153
5,C(ai_tool)[T.llama_meta],1.042747
6,C(ai_tool)[T.other_ai],1.550181
7,C(ai_tool)[T.other_tool],1.004745
8,C(file_role)[T.docs_or_config],1.049316
9,C(file_role)[T.frontend_web],1.288921


## D5. Kiểm tra phần dư (Residual plot)

Vẽ biểu đồ phần dư của mô hình D2 sau khi biến đổi logarit để kiểm tra xem sai số có phân bố đồng đều quanh 0 hay không.

In [6]:
pred = d2_model.get_prediction(d2).summary_frame()
plt.figure(figsize=(8, 5))
sns.scatterplot(x=pred["mean"], y=d2_model.resid, s=12, alpha=0.25)
plt.axhline(0, color="red", ls="--")
plt.xlabel("Gia tri du doan (log_days)")
plt.ylabel("Phan du (Residual)")
plt.title("D5: Bieu do phan du mo hinh D2")
savefig("D5_residual_plot.png")